# Занятие 7, пара 2. Хранилище данных. Слои и витрины


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Чтобы завтра получить тот же отчёт

Таблица почищена, запрос написан. А через неделю пришла новая выгрузка — куда её положить?
Как узнать, какие строки мы отбросили, и пересчитать отчёт, не собирая всё вручную?
Здесь помогают слои: **исходник → очищенное → данные для конкретного вопроса**.

Построим маленькое локальное хранилище. Не нужны сервер и десяток новых терминов.
Исходник сохраним, причины отказа запишем, чистые данные проверим ограничениями, отчёт оформим как VIEW.
SQLite обеспечивает основной путь. В конце есть отдельно проверяемый переход к DuckDB для дальнейшего проекта.

Нужны очистка из 05 и запросы 06; CTE пригодится, но основная работа не требует выполнения 07.1.
Результат — файл учебной базы, воспроизводимая сборка слоёв, витрина и проверка повторного запуска.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Что хранится на каждом слое

**Raw** — как получили, даже если там ошибки. **Clean** — принятые в работу строки по явным правилам.
**Mart, витрина** — таблица или запрос под определённый вопрос. Справочник клиентов связывает события с городом.
В больших системах встречаются STG, ODS, DDS и другие названия; границы зависят от проекта.
В нашей практике трёх слоёв достаточно. Дубликат raw — повод для расследования, а не для потери исходного свидетельства.


In [ ]:
import json
import sqlite3
import tempfile
from pathlib import Path
import pandas as pd
from IPython.display import display

with (DATA_DIR / 'warehouse' / 'demo.json').open(encoding='utf-8') as file:
    source = json.load(file)
raw = pd.DataFrame(source['events'])
customers = pd.DataFrame(source['customers'])
display(raw)
display(customers)


Создадим отдельную папку запуска. `tempfile.mkdtemp(prefix=..., dir=...)` даёт новое уникальное
имя внутри reports; это обычная сохраняемая папка, не временное ядро Jupyter.
Копируем исходник в JSON и открываем файл SQLite. Файл базы останется после закрытия подключения.
`to_sql('raw_events',...)` годится для сырой копии: здесь ещё нет обещания уникальности ключей.


In [ ]:
report_dir = WORKSPACE_DIR / 'reports'
report_dir.mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix='07_warehouse_', dir=report_dir))
(run_dir / 'source.json').write_text(json.dumps(source, ensure_ascii=False, indent=2), encoding='utf-8')
database_path = run_dir / 'practice.sqlite'
connection = sqlite3.connect(database_path)
raw.to_sql('raw_events', connection, index=False, if_exists='replace')
print('Этот запуск:', run_dir)


### 2. Принятые правила и карантин

В нашем источнике стоимость неотрицательна,0 допустим, пропуск остаётся неизвестностью.
Клиент должен быть в справочнике. Точные повторы всей строки удаляем **только из рабочей копии**.
`.isin(...)` проверяет принадлежность списку известных ключей, `~` переворачивает логическую маску.
Для непригодной строки сохраним причину; если причин несколько, отметим обе.


In [ ]:
unique_rows = raw.drop_duplicates().copy()
unique_rows['reason'] = ''
negative = unique_rows['amount'] < 0
unknown_customer = ~unique_rows['customer_id'].isin(customers['customer_id'])
unique_rows.loc[negative, 'reason'] += 'negative_amount;'
unique_rows.loc[unknown_customer, 'reason'] += 'unknown_customer;'
rejected = unique_rows.loc[unique_rows['reason'] != ''].copy()
clean_events = unique_rows.loc[unique_rows['reason'] == '', ['event_id','customer_id','amount']].copy()
display(rejected)
display(clean_events)
print('Raw:',len(raw),'точных повторов:',len(raw)-len(unique_rows),'карантин:',len(rejected),'clean:',len(clean_events))


Было 9 строк: один точный повтор, две строки на разбор, шесть приняты. Сам пропуск стоимости
не удалили. Это правила конкретной задачи, не универсальная чистка любых платежей.
Проверим ключ уже в clean. `assert` остановит выполнение, если обещание нарушено.


In [ ]:
assert clean_events['event_id'].notna().all()
assert not clean_events['event_id'].duplicated().any()
assert len(raw) == (len(raw)-len(unique_rows)) + len(rejected) + len(clean_events)
rejected.to_csv(run_dir / 'rejected.csv', index=False)


### ✏️ Ваш ход — конфликт нельзя назвать точной копией

Пришли строки с ключами a,b,b,c и суммами 10,20,25,0.
Автор отчёта предлагает `drop_duplicates('event_id')`: оставить первое и забыть проблему.
Создайте `my_batch`. Сохраните в `my_conflicts` **все строки с повторным ключом**, а в `my_accepted`
— только строки с однозначным ключом. Исходник не меняйте. Объясните, почему 20 не обоснованнее 25,
и какое дополнительное поле позволило бы выбирать последнюю версию.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.2.1', my_batch, my_conflicts, my_accepted)


**Мой вывод:** …


### 3. Закрепим обещания в базе

`CREATE TABLE` создаёт таблицу по описанию. TEXT/REAL — текст и число; `PRIMARY KEY` требует
уникальный ключ, `NOT NULL` запрещает пропуск, `CHECK` задаёт условие.
SQLite гибок в типах: одного слова REAL недостаточно для всех проверок ввода. Поэтому
`typeof(amount)` проверит фактический тип: integer или real; NULL разрешаем отдельной веткой.

`REFERENCES customers(customer_id)` связывает ключ со справочником. В SQLite проверку внешних
ключей явно включаем через PRAGMA **до начала загрузки**. Сначала завершим запись raw через commit.
`execute` выполняет одну SQL-команду; `executescript` — несколько команд, разделённых точкой с запятой.

`fetchone()` берёт одну строку результата запроса, `[0]` — её первое значение.


In [ ]:
connection.commit()
connection.execute('PRAGMA foreign_keys = ON')
connection.executescript("""
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY NOT NULL,
    city TEXT NOT NULL
);
CREATE TABLE clean_events (
    event_id TEXT PRIMARY KEY NOT NULL,
    customer_id TEXT NOT NULL REFERENCES customers(customer_id),
    amount REAL CHECK(amount IS NULL OR (typeof(amount) IN ('integer','real') AND amount >= 0))
);
""")
print('Внешние ключи включены:', connection.execute('PRAGMA foreign_keys').fetchone()[0])


В таблицу со схемой будем **добавлять**, а не применять `to_sql(..., if_exists='replace')`:
replace удалил бы таблицу вместе с созданными ограничениями.
`.itertuples(index=False, name=None)` даёт строки DataFrame как обычные кортежи.
Заменим пропуск Pandas на Python None, чтобы записать SQL NULL. `executemany` применяет INSERT
к каждой строке; вопросительные знаки — параметры значений, не склейка SQL.


In [ ]:
customer_rows = list(customers.itertuples(index=False, name=None))
event_rows = []
for event_id, customer_id, amount in clean_events.itertuples(index=False, name=None):
    if pd.isna(amount):
        amount = None
    event_rows.append((event_id, customer_id, amount))
connection.executemany('INSERT INTO customers VALUES (?,?)', customer_rows)
connection.executemany('INSERT INTO clean_events VALUES (?,?,?)', event_rows)
connection.commit()


Проверим нарушение на отдельной попытке. Ловим ожидаемый `IntegrityError`, не скрываем
все возможные ошибки. `rollback()` отменяет незавершённую запись; уже подтверждённая загрузка выше сохраняется.
Ограничение не объясняет бизнес-причину само по себе: оно защищает заранее согласованный договор.


In [ ]:
try:
    connection.execute('INSERT INTO clean_events VALUES (?,?,?)', ('bad','A',-1))
    connection.commit()
except sqlite3.IntegrityError as error:
    connection.rollback()
    print('Нарушение замечено:', error)
print('Строк по-прежнему:', connection.execute('SELECT COUNT(*) FROM clean_events').fetchone()[0])


### ✏️ Ваш ход — договор для другой задачи

Отдельная таблица `my_reservations` описывает места в аудитории:
reservation_id — обязательный уникальный текстовый ключ; seats — **целое** число от 1 до 15 включительно.
Напишите CREATE TABLE в своём отдельном подключении `my_connection`. В качестве опоры используйте
разобранные PRIMARY KEY, NOT NULL, typeof и CHECK, но выберите ограничения по новому смыслу.
Проверьте: нормальную запись,0,16,1.5,пропуск и повтор ключа. Должна сохраняться нормальная запись.
Не используйте основное connection и не меняйте clean_events.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.2.2', my_connection)


**Мой вывод:** …


### 4. Витрина — сохранённый вопрос

`CREATE VIEW` сохраняет запрос под именем. Обычный VIEW не хранит отдельную копию результата:
при чтении он обращается к базовым таблицам. Витрину проектируют под вопрос, а не ради нового слоя.
Наш вопрос: сколько событий и известной стоимости приходится на каждый город?
`COUNT(*)` и `COUNT(amount)` снова различаются: неизвестные суммы не становятся нулями.


In [ ]:
connection.execute("""
CREATE VIEW mart_city AS
SELECT customers.city,
       COUNT(*) AS events,
       COUNT(clean_events.amount) AS known_amounts,
       SUM(clean_events.amount) AS known_total,
       AVG(clean_events.amount) AS known_mean
FROM clean_events
JOIN customers ON clean_events.customer_id = customers.customer_id
GROUP BY customers.city
""")
connection.commit()
city_mart = pd.read_sql_query('SELECT * FROM mart_city ORDER BY city', connection)
display(city_mart)


INNER JOIN здесь не теряет событие: мы включили и проверили внешний ключ, он обязательный,
а справочник уникален. Если этих предпосылок нет, прежний опыт с left/inner снова важен.
Для Севера 4 события, но известны 3 стоимости. Называем сумму **известной**, не общей доказанной выручкой.


### ✏️ Ваш ход — восстановить отчёт из частичных итогов

В другой системе исходные события вам не передали. Есть только итоги клиентов:

| city | customer_id | known_total | known_count |
|---|---|---|---|
| Север | A | 100 | 2 |
| Север | C | 140 | 1 |
| Юг | B | 270 | 2 |

Коллега нашёл среднюю стоимость каждого клиента и усреднил их по городу. Для Севера вышло 95.
Нужна средняя **на событие с известной стоимостью**, а не на клиента. Напишите функцию
`my_city_report(client_summary)`, которая возвращает DataFrame с колонками `city, known_mean`
по городам в алфавитном порядке. Используйте знакомые группировку, суммы и деление; исходные
`clean_events` читать нельзя — здесь их условно нет. Счётчики во входе положительные,
итоги клиентов не перекрываются, пропуски уже исключены из known_count.

Затем у A измените одновременно сумму на 200 и число событий на 4: его среднее по-прежнему 50.
Должен ли итог города остаться прежним? Объясните изменение веса. Какого поля не хватило бы,
если бы вам прислали только среднее клиента без суммы и количества?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.2.3', my_city_report)


**Мой вывод:** …


### 5. Повторный запуск должен быть предсказуемым

Повторный INSERT тех же ключей сейчас будет остановлен — это защита от случайного удвоения.
Для **нашего небольшого полного снимка** выберем пересборку clean целиком внутри транзакции.
`with connection:` подтверждает действия при успехе и откатывает их при исключении.
DELETE без WHERE удаляет все строки указанной таблицы: применяем его только к своему
восстанавливаемому учебному clean, raw и справочник не трогаем. Для больших рабочих систем
и неполных выгрузок стратегию обновления выбирают отдельно.

`pd.testing.assert_frame_equal(before, after)` сравнивает две таблицы и останавливает опыт, если они различаются. Здесь это содержательная проверка: повторный запуск не должен незаметно удвоить показатели.


In [ ]:
before = pd.read_sql_query('SELECT * FROM mart_city ORDER BY city',connection)
with connection:
    connection.execute('DELETE FROM clean_events')
    connection.executemany('INSERT INTO clean_events VALUES (?,?,?)',event_rows)
after = pd.read_sql_query('SELECT * FROM mart_city ORDER BY city',connection)
pd.testing.assert_frame_equal(before,after)
after.to_csv(run_dir / 'mart_city.csv',index=False)
connection.close()
reopened = sqlite3.connect(database_path)
display(pd.read_sql_query('SELECT * FROM mart_city ORDER BY city',reopened))
reopened.close()
print('Файл базы и результат сохранены в',run_dir)


### 6. Переход к DuckDB для проекта

SQLite помог выполнить опыт без установки. В критериях будущей ЛР 4 сейчас указан DuckDB,
поэтому отдельно покажем тот же путь в нём. Это другой движок, а не название файла SQLite.
Устанавливать его посреди пары не нужно: пример можно повторить в подготовленном окружении.
Дальнейшая работа проекта потребует полного описания слоёв, ключей и витрин, не одного import.

`duckdb.connect` открывает базу. `register` делает DataFrame доступным для SQL в текущем подключении;
это ещё не сохранённая таблица. `CREATE TABLE ... AS SELECT` материализует снимок.
Текст запроса витрины тот же, но ограничения таблицы надо задавать отдельно — CTAS их не переносит.

В запросе ниже `USING(customer_id)` — короткая форма соединения по одноимённым ключам;
равносильна разобранному ON для этих непустых ключей. `fetchdf()` возвращает DataFrame.


In [ ]:
try:
    import duckdb
except ImportError:
    duckdb = None

if duckdb is None:
    print('DuckDB не установлен: основной опыт выполнен на SQLite; переход ещё нужно запустить отдельно.')
else:
    duck_path = run_dir / 'practice.duckdb'
    duck = duckdb.connect(str(duck_path))
    duck.register('raw_frame',raw)
    duck.register('clean_frame',clean_events)
    duck.register('customer_frame',customers)
    duck.execute('CREATE TABLE raw_events AS SELECT * FROM raw_frame')
    duck.execute('CREATE TABLE clean_events AS SELECT * FROM clean_frame')
    duck.execute('CREATE TABLE customers AS SELECT * FROM customer_frame')
    duck.execute("""CREATE VIEW mart_city AS
        SELECT customers.city, COUNT(*) AS events, COUNT(amount) AS known_amounts,
               SUM(amount) AS known_total, AVG(amount) AS known_mean
        FROM clean_events JOIN customers USING(customer_id) GROUP BY customers.city""")
    duck_result = duck.execute('SELECT * FROM mart_city ORDER BY city').fetchdf()
    pd.testing.assert_frame_equal(after,duck_result,check_dtype=False)
    duck.close()
    print('Снимок DuckDB сохранён, результаты совпали:',duck_path)


**Следующий шаг проекта:** набросайте свою цепочку raw→clean→витрина и укажите смысл строки
на каждом шаге. Отдельно назовите проверки, которые можно закрепить ограничениями,
и решения, которые требуют знания предметной области. ЛР 4 будет выдана с анализом/графиками позже;
баллы и сроки сегодня не меняются.

**Резерв 10–15 минут:** добавьте в отдельную копию базы новый допустимый клиент и событие.
Проверьте, что VIEW изменился без пересоздания. Затем объясните, почему экспортированный CSV
сам не обновился. Сохраните опыт отдельно, основной файл результата не заменяйте.

Справочники: [внешние ключи SQLite](https://sqlite.org/foreignkeys.html),
[Python-интерфейс DuckDB](https://duckdb.org/docs/stable/clients/python/overview).
Чтение необязательно; полезнее сначала понять, что именно сохранено в вашем файле.


**Куда почитать дальше, если тема зацепила:** Martin Kleppmann,
[Designing Data-Intensive Applications](https://dataintensive.net/).
Это книга о выборе способов хранения и обработки данных, не справочник команд SQLite.
Сейчас достаточно посмотреть оглавление и найти знакомые вопросы про модели данных и надёжность;
читать целиком к следующей паре не требуется.
